# Notebook 09 -- facet_similarity

**Asks: score each answer by how close it sits to each facet reference set.**

For every answer and every one of the nine facets, this notebook produces one
number: the mean *cosine similarity* between that answer and the facet's set of
reference passages. Cosine similarity is the angle between two passages once an
embedding model (a model that turns a passage into a long list of numbers) has
placed both in the same numeric space -- 1.0 means the same direction, 0.0 means
unrelated. Close to the desert reference set means the answer talks the way the
desert passages talk.

**Where this is registered.** Prereg 6.7 names two measures that do not use a
language model and are scored alongside the ensemble: a word list, and
"similarity to short reference passages for each facet, computed with more than
one embedding model". This notebook is the second of them. The method it
implements is MEASUREMENT_DESIGN.md section 5 -- per-facet reference-set mean
cosine, computed per `item_type`, with leave-one-out. That no language model is
consulted anywhere in it is the whole reason it can serve as a check on one.

This is Method 2 of three. The nine facets are scored a word-list way (notebook
01), a language-model way (the model ensemble, merged in notebook 07), and this
embedding way, and the three are then compared.

**What this notebook is not.** It is secondary convergence, and it gates nothing.
The registered gate has exactly two clauses -- the models agree with each other at
an intraclass correlation (ICC, a measure of how much independent raters scoring
the same things agree) of at least .60, and each facet correlates at least .30
with its matching survey scale -- and neither clause is computed from an
embedding. Nothing here can pass or fail a facet.

**In** -- three files for every encoder notebook 08 was run with:
`embeddings_<key>.npy` (one row per answer), `embeddings_<key>_seed.npy` (one row
per reference passage) and `embeddings_<key>_meta.json` (what each row of each one
is); plus `embedding_seed_pools.json` (notebook 04),
`gold_split_assignment.csv` (notebook 03), `openends_long.csv` (notebook 00).
**Out** -- `facet_similarity.csv`, `embedding_model_sensitivity.csv`.


## Config

In [1]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "09"                     # this notebook's number
NB_NAME     = "facet_similarity"       # this notebook's job, in two words
NB_ASKS     = "Score each answer by how close it sits to each facet reference set."

## Setup

In [2]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 09 -- facet_similarity
  Score each answer by how close it sits to each facet reference set.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  this folder is empty: notebook 00 is the place to start.


## Load

In [3]:
# === IMPORTS AND THE FACET REGISTER ===
# WHAT: the numeric stack this notebook needs, then the nine-facet vocabulary that
#       every column name below is built from.
import pandas as pd
import numpy as np
import glob
import warnings
# Only the two categories that are pure noise here are silenced: deprecation and
# future-behaviour notices from pandas and numpy. RuntimeWarning is deliberately
# LEFT AUDIBLE, because scipy raises one of those (ConstantInputWarning, which is a
# subclass of RuntimeWarning) when a similarity column is flat and a rank
# correlation on it is meaningless. That warning is a signal this notebook is read
# for -- it is a sparse facet announcing that it orders nothing -- so a blanket
# filterwarnings("ignore") would suppress the very evidence the coverage floor
# below exists to report.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
# spearmanr carries the rank-order half of the sensitivity report (prereg 6.7).
# The report is deliberately RANK-order and never linear, because the absolute
# cosines a linear correlation would work on are encoder-dependent by construction.
from scipy.stats import spearmanr

# Wide display so a nine-facet describe() prints without columns being elided, and
# fixed point so cosines (which live in roughly [0.2, 0.9]) are legible.
pd.set_option("display.max_columns", 50)
np.set_printoptions(precision=4, suppress=True)

# The nine facets. Six retributive -- four principled (desert, proportionality,
# censure, moral_balance) and two vengeful (revenge, suffering) -- plus the three
# consequentialist goals, which are scored so that retributive language is measured
# against a full menu of punishment justifications rather than in a vacuum (prereg
# 5.15). They are NEVER summed into a retributivism total, and there is no
# prosocial-minus-dark axis anywhere in this notebook: the object of measurement is
# a nine-way distinction, not a valence.
# This ORDER is load-bearing. It fixes the column order of every similarity frame,
# the melt order in the export, and the row order of the sensitivity table the
# convergence notebook joins to. Change it and nothing errors; the files simply
# stop lining up.
FACETS = ["desert", "proportionality", "censure", "moral_balance",
          "revenge", "suffering", "deterrence", "incapacitation", "rehabilitation"]

# Sparse facets: the embedding leg is reported for them only where it clears the
# coverage floor further down. These four fire on roughly 1 to 2.5 per cent of
# responses (proportionality about 1.0 per cent; 0.3 per cent on the backward
# item), so their similarity distribution can be near-degenerate -- every response
# sitting about the same distance from a handful of exemplars. A sparse facet is
# still REPORTED; what it loses is standing as an independent convergence leg, on
# the same reasoning that disqualifies the word list below 5 per cent coverage:
# agreement computed on a handful of positives is noise, and its silence reads
# falsely as "no independent contradiction" when it is "no measurement".
SPARSE_FACETS = ["proportionality", "censure", "moral_balance", "revenge"]
print("Imports complete; nine facets registered.")


Imports complete; nine facets registered.


In [5]:
# === LOAD 1 of 2: the corpus, and the one-row-per-answer contract ===
# WHAT: the response index -- which row of each embedding matrix is which answer.
# WHY THE RESPONSE INDEX MATTERS MORE THAN IT LOOKS: a .npy matrix carries no row
#       labels. openends_long.csv IS the row order of every embedding matrix
#       notebook 08 wrote, so this file is what turns row 1,742 of a matrix back
#       into a person and a question. Load a different file here and every
#       similarity below lands on the wrong answer, silently.
open_path = need("openends_long.csv", produced_by="00")
df = pd.read_csv(open_path)
assert {"response_id", "item_type", "text"}.issubset(df.columns), \
    "openends_long.csv must have response_id, item_type, text"

# THE CONTRACT, ENFORCED HERE AND NOT ASSUMED: one row per
# (response_id, item_type). Notebook 00 checks it for the file it writes, and
# nothing between there and here re-checks it, so it is checked again on the file
# as read -- a hand-edited or twice-appended corpus is exactly the case a comment
# about an upstream guarantee would miss. A duplicated pair would fan out in the
# reshape at the end, raise the row count of facet_similarity.csv, and raise the
# sample size of every analysis downstream while the n printed beside it stayed the
# same.
_dups = df[df.duplicated(["response_id", "item_type"], keep=False)]
assert _dups.empty, (
    f"openends_long.csv has {len(_dups)} rows sharing a (response_id, item_type) "
    f"pair, for example "
    f"{_dups[['response_id', 'item_type']].head(3).to_dict('records')}. One row "
    f"per answer is the key every join in this notebook uses. Re-run notebook 00.")

if RUN_MODE == "smoke":
    # Trim by PARTICIPANT, never by row. Every person contributes three rows, one
    # per open-ended question, and notebooks 00, 03 and 08 all cut on participants.
    # A head(20) here would keep 20 ROWS, which is six and two thirds of a person.
    # It would also cut the corpus at a different point from notebook 08's smoke
    # run, so the row-key check further down would reject every encoder and smoke
    # mode could not complete at all.
    df = df[df["response_id"].isin(
        pd.unique(df["response_id"])[:20])].reset_index(drop=True)
    print("  smoke mode: first 20 participants only, a plumbing test not a result")

# The three open-ends are scored SEPARATELY and never pooled (prereg 5.15). The
# sentence justification and the ranking explanation look BACKWARD, where desert
# and proportionality language sits; the accomplishment item looks FORWARD, where
# deterrence, incapacitation and rehabilitation language sits. Combining them would
# mix the two registers whose contrast is itself a measure (E6). An unexpected
# item_type would not error downstream -- it would silently create a fourth
# reference set inside facet_similarity -- which is why it stops here.
assert set(df.item_type.unique()) <= {"ranking", "backward", "forward"}, \
    f"Unexpected item_type values: {set(df.item_type.unique())}"
print(f"  responses: {len(df)} rows, {df.response_id.nunique()} participants, "
      f"{df.item_type.value_counts().to_dict()}")
# Expect 2,844 rows in full mode: 948 participants retained after the preregistered
# exclusions (1,015 raw -> 965 finished -> 958 -> 948) times three open-ends, with
# blanks kept as rows. A count built on 1,015 means the analytic-sample restriction
# did not happen upstream, and these scores will not line up with the ensemble
# scores they are meant to be compared with.


  [ok]   openends_long.csv  (written unknown time)
  responses: 2844 rows, 948 participants, {'ranking': 948, 'backward': 948, 'forward': 948}


In [8]:
# === LOAD 2 of 2: the frozen split and the reference pools ===
split_path = need("gold_split_assignment.csv", produced_by="03")
pools_path = need("embedding_seed_pools.json", produced_by="04")

split = pd.read_csv(split_path)
# The split is frozen PER ITEM, so this file holds one row per
# (response_id, item_type) and item_type is part of its key, not decoration. The
# export at the end joins on both keys: a response_id-only join fans the rows out
# and can hand an item frozen into SEED a TEST label.
assert {"response_id", "item_type", "gold_split"}.issubset(split.columns), \
    "gold_split_assignment.csv must have response_id, item_type, gold_split"
# ENFORCED HERE, for the two things below that depend on it: the audit lookup in
# the next cell, which would silently keep the LAST of two conflicting labels for
# the same answer, and the merge in the export, where a duplicated key multiplies
# similarity rows. This is the right-hand frame of the only merge in this notebook,
# and its uniqueness is checked on the file as read rather than taken on trust.
_sdups = split[split.duplicated(["response_id", "item_type"], keep=False)]
assert _sdups.empty, (
    f"gold_split_assignment.csv has {len(_sdups)} rows sharing a "
    f"(response_id, item_type) pair, for example "
    f"{_sdups[['response_id', 'item_type']].head(3).to_dict('records')}. The split "
    f"is frozen one row per answer. Re-run notebook 03, or restore the posted "
    f"assignment file.")
print(f"  frozen split: SEED={(split.gold_split == 'SEED').sum()} rows, "
      f"TEST={(split.gold_split == 'TEST').sum()} rows")
# The split covers the drawn strata sample only (roughly 250 to 300 answers), not
# the whole corpus, so most rows of the export below carry neither label. That is
# expected, and it is spelled out as NOT_IN_SPLIT rather than left blank.

with open(pools_path) as f:
    _pools_json = json.load(f)
print("  reference pool file read; unwrapped in the next cell.")


  [ok]   gold_split_assignment.csv  (written unknown time)
  [ok]   embedding_seed_pools.json  (written unknown time)
  frozen split: SEED=154 rows, TEST=154 rows
  reference pool file read; unwrapped in the next cell.


## Do

In [9]:
# === THE REFERENCE POOLS: what they are, and where notebook 04 got them ===
# WHAT: unwraps the pool file and prints the provenance recorded inside it.
# WHY PRINT PROVENANCE AT ALL: the whole worth of this leg is that it is
#       independent of what it is later compared with. Reference passages may come
#       only from the SEED half of the frozen split, and a reader has to be able to
#       SEE that from this notebook's own output rather than take it on trust from
#       a filename. That is why the provenance travels inside the file.
# CONTRACT: notebook 04 writes a wrapper object -- the facet mapping under a
#       "pools" key, beside the provenance fields recorded with it. A file that is
#       already the bare mapping is accepted too, so a hand-built pool file runs.
if isinstance(_pools_json, dict) and "pools" in _pools_json:
    seed_pools = _pools_json["pools"]
    _prov = {k: v for k, v in _pools_json.items() if k != "pools"}
else:
    seed_pools = _pools_json
    _prov = {}

print("  provenance recorded in embedding_seed_pools.json:")
if _prov:
    # The run-level facts first, then the routes the passages came in by. The line
    # that matters most is test_rows_used: it must read 0.
    for k in ["source", "random_seed", "built_by", "built_at", "run_mode", "human_gold"]:
        if k in _prov:
            print(f"    {k}: {_prov[k]}")
    for k, v in _prov.get("routes", {}).items():
        print(f"    route {k}: {v}")
    _sp = _prov.get("split", {})
    if _sp:
        print(f"    split: SEED rows={_sp.get('seed_rows')}, "
              f"TEST rows={_sp.get('test_rows')}, "
              f"TEST rows used={_sp.get('test_rows_used')}")
        if _sp.get("codebook_anchors_dropped_as_TEST"):
            print(f"    codebook anchors dropped for landing in TEST: "
                  f"{_sp['codebook_anchors_dropped_as_TEST']}")
else:
    print("    (none recorded -- this file did not come from notebook 04)")

# Notebook 04 empties any pool that falls below the minimum size rather than
# letting a centroid rest on two or three responses. Those facets carry no
# embedding measurement at all, and this notebook says so in as many words --
# "not run" -- rather than reporting them as a blank or as a failure.
for f, st in _prov.get("facet_status", {}).items():
    if st.get("status") != "ok":
        print(f"    {f}: NOT RUN -- {st.get('why', st.get('status'))}")

# Every key must be one of the nine registered facets. Handed the wrapper object
# instead of the mapping, every key would be skipped below, the reference frame
# would come back empty, and all nine similarity columns would be NaN -- a failure
# that reads like a null result rather than like a broken load. Hence a stop here.
_unknown = [k for k in seed_pools if k not in FACETS]
assert not _unknown, f"embedding_seed_pools.json has non-facet keys: {_unknown}"

# A facet with no pool is announced, not fatal: it simply carries no embedding
# measurement, and the sensitivity table reports it as such.
_missing = [f for f in FACETS if f not in seed_pools]
if _missing:
    print(f"  WARNING: no reference pool for {_missing} -- those facets get no similarity.")
print(f"  pools loaded for: {sorted(seed_pools.keys())}")

  provenance recorded in embedding_seed_pools.json:
    source: codebook anchor passages (MEASUREMENT_DESIGN.md section 4) plus word-list-positive responses from outside the TEST half of the frozen split; no ensemble output was consulted; TEST-half responses used: 0; human-coded exemplars used: 0
    random_seed: 20260629
    built_by: 04_reference_pools
    built_at: 2026-08-13T22:09:53
    run_mode: full
    human_gold: False
    route codebook_anchor: author-written anchor passages from the codebook, quoted as the codebook prints them, independent of the ensemble by construction
    route wordlist_non_test: responses the nine-facet word list scored positive, ranked low on the opposing facets, drawn from everywhere except the TEST half
    split: SEED rows=154, TEST rows=154, TEST rows used=0
  pools loaded for: ['censure', 'desert', 'deterrence', 'incapacitation', 'moral_balance', 'proportionality', 'rehabilitation', 'revenge', 'suffering']


In [10]:
# === FLATTEN THE POOLS INTO ONE REFERENCE INDEX ===
# The pools arrive as facet -> list of exemplar dicts, each carrying at least
# response_id, item_type and text. Flattening them into a single table is what lets
# ONE matrix multiply serve all nine facets: the passages were embedded once per
# encoder in notebook 08, and the scoring function slices the resulting matrix with
# facet masks. This flattening reproduces the order notebook 08 embedded them in,
# and the two are compared below -- it is a check on the vectors, not the source of
# their labels.

def _pool_index(seed_pools):
    """Flatten seed_pools -> DataFrame(facet, response_id, item_type, route).

    The cell above unwraps notebook 04's wrapper object and asserts that every key
    is a registered facet, so what arrives here is the facet mapping itself. The
    non-facet branch below is the guard for a pool dict built by hand and handed
    straight to this function.
    """
    rows = []
    for facet, exemplars in seed_pools.items():
        # Only the nine registered facets get a reference set; anything else is a
        # typo or a hand-added key and must not become a tenth similarity column.
        if facet not in FACETS:
            print(f"  WARNING: pool facet {facet!r} not in FACETS; skipping")
            continue
        for ex in exemplars:
            # .get() rather than [] -- a pool entry written without item_type then
            # surfaces as a value matching no response's item_type (the passage
            # simply drops out of every set) instead of raising here.
            rows.append({"facet": facet,
                         "response_id": ex.get("response_id"),
                         "item_type": ex.get("item_type"),
                         "route": ex.get("route", "unknown")})
    # Columns are declared explicitly so that an empty pool still returns a table
    # carrying a facet column: the count below then prints 0 and the similarity
    # comes back all-NaN, which is legible, where a column-less frame would raise.
    pi = pd.DataFrame(rows, columns=["facet", "response_id", "item_type", "route"])
    # Read this line as the health check for the whole notebook: expect 8 to 15
    # passages per facet across 9 facets, so roughly 100 to 140 rows. A count well
    # under that means the SEED half barely contains those facets, and a similarity
    # computed against a thin pool rests on a handful of passages.
    print(f"  reference passages: {len(pi)} across {pi.facet.nunique()} facets")
    return pi

pool_index = _pool_index(seed_pools)
# One passage may legitimately serve TWO facets -- the mixed-motive exemplars are
# placed in both the principled and the vengeful pool on purpose -- so the key here
# is the triple, not the pair. What must not happen is the SAME passage twice in
# ONE facet: notebook 04 keeps a `seen` set per pool to prevent it, and a pool file
# edited by hand has no such protection. A repeated passage would be counted twice
# in that facet's mean cosine, quietly doubling one passage's weight.
_pdups = pool_index[pool_index.duplicated(["facet", "response_id", "item_type"],
                                          keep=False)]
assert _pdups.empty, (
    f"embedding_seed_pools.json lists the same passage twice in one facet: "
    f"{_pdups.head(3).to_dict('records')}. Each passage may appear in more than "
    f"one facet, but only once within a facet. Re-run notebook 04.")
if len(pool_index):
    print(pool_index.groupby("facet").size().to_string())
for f in FACETS:
    if not (pool_index.facet == f).any():
        print(f"  {f}: no reference passages, so its similarity is NOT RUN.")


  reference passages: 133 across 9 facets
facet
censure            15
desert             15
deterrence         15
incapacitation     15
moral_balance      14
proportionality    15
rehabilitation     15
revenge            14
suffering          15


In [11]:
# === AUDIT: the reference passages came only from SEED ===
# WHAT: checks every reference passage against the frozen split and says out loud
#       where each one came from.
# WHY IT IS WORTH A CELL: leave-one-out (below) removes only a response's
#       similarity to ITSELF. It cannot remove the shared signal running through
#       the rest of a stratum, so leave-one-out alone would leave any later
#       convergence number partly circular. The frozen split is what closes that,
#       and it is mandatory rather than tidy (design section 8; anti-circularity
#       rule 4: never report a convergence number for a stream on the responses
#       that seeded that stream). This cell is the mechanical check that the rule
#       actually held, rather than a sentence in a methods section claiming it did.
_lookup = {(str(r.response_id), str(r.item_type)): r.gold_split
           for r in split.itertuples(index=False)}
# Every passage is looked up in gold_split_assignment.csv itself rather than
# trusting the gold_split label notebook 04 wrote beside it. A label copied from
# the same place it is being checked against proves nothing.
_counts = {"SEED half": 0, "TEST half": 0, "outside the split sample": 0}
_offenders = []
for _, r in pool_index.iterrows():
    lab = _lookup.get((str(r["response_id"]), str(r["item_type"])))
    if lab == "SEED":
        _counts["SEED half"] += 1
    elif lab == "TEST":
        _counts["TEST half"] += 1
        _offenders.append((r["facet"], r["response_id"], r["item_type"]))
    else:
        # A response outside the stratified split sample is in neither half, so it
        # cannot carry TEST-half signal into a reference set. Codebook anchor
        # passages are the usual occupants of this line: they are real quotes from
        # this corpus, which is why they bring its register with them.
        _counts["outside the split sample"] += 1
print("  where the reference passages came from:")
for k, v in _counts.items():
    print(f"    {k}: {v}")
print("  by route: " + ", ".join(f"{k}={v}" for k, v in
                                 pool_index.route.value_counts().items()))
if _offenders:
    print("  WARNING: reference passages drawn from the TEST half. This breaks")
    print("  anti-circularity rule 4, and any TEST-half convergence number computed")
    print("  from these vectors would be partly self-agreement. Fix notebook 04.")
    for o in _offenders[:10]:
        print(f"    {o}")
else:
    print("  OK: no reference passage comes from the TEST half.")

  where the reference passages came from:
    SEED half: 3
    TEST half: 0
    outside the split sample: 130
  by route: wordlist_non_test=123, codebook_anchor=10
  OK: no reference passage comes from the TEST half.


In [12]:
# === COSINE PLUMBING: L2 ROW NORMALIZATION ===
# Normalizing a vector to length 1 is what makes a dot product BE a cosine. The
# encoders in notebook 08 are already asked for unit-length rows, so on their
# output this is a no-op. It is applied anyway, for two reasons: it makes
# `responses @ references.T` a cosine matrix BY CONSTRUCTION rather than by
# trusting a keyword argument in another notebook, and the Voyage API makes no
# unit-length guarantee at all.

def normalize_rows(emb_matrix):
    """L2-normalize each row so that cosine == dot product."""
    norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
    # Clip the denominator: an all-zero embedding (possible for a degenerate input)
    # would otherwise divide by zero and spray NaN through the whole matrix.
    return emb_matrix / np.clip(norms, 1e-8, None)

print("Normalization helper defined.")


Normalization helper defined.


### What the score is, and the three properties that must hold

A response's score on a facet is the **mean cosine to that facet's reference
set**, computed **separately per `item_type`**, with **leave-one-out**. That is
what MEASUREMENT_DESIGN.md section 5 specifies. The three properties are enforced
in the function below rather than filtered for afterwards, because a downstream
filter cannot undo a number that was computed the wrong way.

**1. Per `item_type` (prereg 5.15).** A backward-looking justification is compared
only with backward-looking reference passages. Pooling would let the forward
item's register -- systematically different, not merely noisier -- set the
baseline for the backward item and the other way round, flattening the very
contrast E6 exists to measure.

**2. Leave-one-out.** Where a response is itself one of a facet's reference
passages, its cosine with itself is 1.0 by construction and would inflate that
facet's score for that response. Dropping the reference passage that shares its
`response_id` removes that self-similarity exactly. Leave-one-out is necessary but
NOT sufficient for independence: it cannot remove the signal shared across a
stratum, which is what the frozen SEED/TEST split is for.

**3. NaN, never 0.0, wherever a score is undefined** -- no passages for the facet,
none inside this `item_type`, or leave-one-out emptied the set. A 0.0 would read
downstream as "maximally dissimilar", which is a claim; NaN is the absence of one,
and every consumer below (`describe`, `spearmanr`, the variance check) drops NaN
rather than averaging it in.

### Two choices behind the reference sets

**The mean over the set, not the max.** The max is the nearest single passage,
which makes a response's score hostage to one passage and to whichever reference
happens to share its vocabulary. The mean over 8 to 15 passages is what
MEASUREMENT_DESIGN.md section 5 specifies and is far less sensitive to any single
one.

**Real passages, not one composed prototype.** A prototype written for the purpose
encodes the analyst's own theory of the facet, and every similarity score
downstream then measures that theory instead of the corpus. The reference sets are
built from the codebook's anchor passages -- which are real quotes from this
corpus -- and from dictionary-positive responses in the SEED half, so they carry
the corpus's own register: short, blunt, ungrammatical, mixed-motive. Mixed-motive
passages are deliberately placed in BOTH the principled and the vengeful sets, so
the anchors reflect the real co-occurrence E4 exists to measure rather than an
idealised separation the corpus does not contain.


In [13]:
# === THE SCORING FUNCTION ===
# IMPLEMENTS MEASUREMENT_DESIGN.md section 5. Defined once and called identically
# for every encoder, so a difference in the sensitivity table is attributable to
# the encoder rather than to two divergent copies of the code.
def facet_similarity(resp_emb, resp_ids, resp_items, seed_frame, seed_emb):
    """Per-facet reference-SET mean cosine, computed PER item_type, with leave-one-out.

    resp_emb   : (n_resp, dim) row-normalized response embeddings
    resp_ids   : response_id for each row of resp_emb
    resp_items : item_type for each row of resp_emb
    seed_frame : DataFrame(facet, response_id, item_type, ...), one row per
                 reference passage, in the order notebook 08 embedded them
    seed_emb   : (n_seed, dim) row-normalized reference embeddings, row-aligned to
                 seed_frame

    Returns a DataFrame of [response_id, item_type] plus one <facet>_sim column.
    """
    # Coerce to arrays so that masking and positional indexing behave the same
    # whether the caller passed a Series, a list or an ndarray.
    resp_ids = np.asarray(resp_ids)
    resp_items = np.asarray(resp_items)
    out = pd.DataFrame({"response_id": resp_ids, "item_type": resp_items})
    # LEAVE-ONE-OUT COMPARES IDS AS TEXT, and that is not a formality. The
    # reference ids come from a JSON metadata file, which notebook 08 writes with
    # default=str; the response ids come from a CSV, where pandas types a numeric
    # id column as int64. Compared raw, 4711 never equals "4711", so a response
    # that IS its own facet's reference passage would keep its self-cosine of 1.0
    # in the mean instead of having it dropped. The OUTPUT keeps the original ids,
    # so the merge at the end still meets the split file on its own types.
    resp_key = resp_ids.astype(str)
    seed_key = seed_frame["response_id"].astype(str).values
    seed_it = seed_frame["item_type"].values
    # ONE matrix multiply for the whole corpus against the whole reference bank.
    # Both sides are row-normalized, so every entry is already a cosine; everything
    # after this line is masking and averaging, with no other distance measure.
    sim_full = resp_emb @ seed_emb.T

    for facet in FACETS:
        # Start every response at NaN. A cell is filled only where a real,
        # non-empty, item_type-matched, leave-one-out-surviving reference set exists.
        col = np.full(len(resp_ids), np.nan)
        fmask = (seed_frame["facet"].values == facet)
        # No passages at all for this facet, so the column stays NaN and the facet
        # simply has no embedding measurement. That is reported, not patched.
        if fmask.sum() == 0:
            out[f"{facet}_sim"] = col
            continue
        for it in np.unique(resp_items):
            set_mask = fmask & (seed_it == it)      # per item_type reference set
            # This facet has no passage in this item_type: leave those responses
            # NaN rather than borrowing across the backward/forward line.
            if set_mask.sum() == 0:
                continue
            set_idx = np.where(set_mask)[0]         # columns of sim_full to average
            r_idx = np.where(resp_items == it)[0]   # rows scored on this pass
            for ri in r_idx:
                # leave-one-out: drop the reference sharing this response's id
                loo = set_idx[seed_key[set_idx] != resp_key[ri]]
                # A one-passage set that IS this response leaves nothing to score
                # against: NaN, not a self-similarity of 1.0.
                if len(loo) == 0:
                    continue
                # The score of record for this response on this facet.
                col[ri] = sim_full[ri, loo].mean()
        out[f"{facet}_sim"] = col
    return out

print("Scoring function defined (reference-set mean cosine; per item_type; leave-one-out).")


Scoring function defined (reference-set mean cosine; per item_type; leave-one-out).


In [14]:
# === WHICH ENCODERS RAN, AND WHAT EACH ONE LEFT BEHIND ===
# WHAT NOTEBOOK 08 WRITES, three files per encoder it is run with:
#   embeddings_<key>.npy       the CORPUS block -- one row per answer, in
#                              openends_long.csv order, already unit length
#   embeddings_<key>_seed.npy  the REFERENCE block -- one row per reference passage,
#                              embedded in the same run by the same checkpoint
#   embeddings_<key>_meta.json what each row of each matrix is: how many rows in
#                              each, the width, the pinned checkpoint, a key for
#                              every corpus row, a facet label for every reference
#                              row, and the provenance of the pools it embedded
# WHY THE TWO MATRICES ARE SEPARATE FILES: notebook 08 caches the corpus block,
#   because a full encode costs GPU minutes or API spend, and re-embeds the
#   reference block on EVERY run, because notebook 04 can rebuild the pools at any
#   time and a stale reference block is the failure no shape check could catch. The
#   pair is written together by one checkpoint in one run. This notebook checks
#   that pairing again below rather than taking it on trust.
#
# The encoders this chain knows about, in notebook 08's own order: the required leg
# first, then the optional ones. Reading the roster rather than the folder is what
# stops a filename inventing an encoder -- embeddings_bge_seed.npy also matches
# embeddings_*.npy, and slicing a name out of it would produce an encoder called
# "bge_seed", hand that invented name to need(), and print an instruction to delete
# the reference vectors and re-run notebook 08 with an ENCODER_KEY its own roster
# would reject.
ENCODER_KEYS  = ["bge", "mpnet", "voyage"]
# BGE is the required encoder. It is open-weight, so the whole embedding leg can be
# reproduced without an API account or a paid key, and it comes from a third maker:
# prereg 6.7 asks for more than one embedding model, and the design pins three from
# three different makers so that a shared-vendor artefact cannot present itself as
# cross-model agreement. "Required" still does not mean "a gate" -- see the top.
PRIMARY_MODEL = "bge"

found = [k for k in ENCODER_KEYS if os.path.exists(f"embeddings_{k}.npy")]
# An embedding file whose key is not in the roster is NAMED and ignored, never
# loaded. A fourth encoder added to notebook 08 has to be added to ENCODER_KEYS
# here as well; until it is, its numbers are visibly absent rather than silently
# half-present.
_stray = sorted({os.path.basename(p)[len("embeddings_"):-len(".npy")]
                 for p in glob.glob("embeddings_*.npy")}
                - set(ENCODER_KEYS) - {f"{k}_seed" for k in ENCODER_KEYS})
if _stray:
    print(f"  embedding files with no roster entry, ignored: {_stray}")
    print("  (add the key to ENCODER_KEYS above if notebook 08 now writes it)")
print(f"  encoders present: {found if found else '(none)'}")

# The row keys openends_long.csv defines, in its row order. Every corpus matrix has
# to carry exactly this list in exactly this order, or it describes other people.
corpus_keys = [f"{r}|{i}" for r, i in
               zip(df.response_id.astype(str), df.item_type.astype(str))]


  encoders present: ['bge', 'mpnet', 'voyage']


In [15]:
# === LOAD EACH ENCODER'S THREE FILES, AND CHECK THEY DESCRIBE THIS CORPUS ===
# WHY THE CHECKS ARE NOT OPTIONAL: a .npy carries no index. A matrix left over from
# a different corpus -- a pilot export, a differently filtered openends_long.csv, a
# smoke run -- would still multiply cleanly and return plausible similarities
# silently attached to the wrong people. The recorded row keys are the only
# alignment check there is. Where one fails we throw the MATRIX away and re-run
# notebook 08; we never re-filter the corpus to make a file fit.
loaded, problems = {}, {}
for m in found:
    rp = need(f"embeddings_{m}.npy", produced_by="08")
    sp = need(f"embeddings_{m}_seed.npy", produced_by="08", optional=True)
    mp = need(f"embeddings_{m}_meta.json", produced_by="08", optional=True)
    if mp is None or sp is None:
        # Without the metadata the .npy is an anonymous pile of numbers: nothing
        # says which row is whose answer. Without the reference matrix there is
        # nothing to take a cosine against. Guessing is exactly the thing that
        # would not look broken, so the encoder is skipped instead.
        problems[m] = ("its metadata file is missing, so its rows cannot be "
                       "identified" if mp is None else
                       "its reference-passage matrix is missing, so there is "
                       "nothing to score against")
        continue
    with open(mp) as fh:
        meta = json.load(fh)
    A, S = np.load(rp), np.load(sp)
    n_c, n_s = int(meta.get("n_corpus", -1)), int(meta.get("n_seed", -1))
    if A.shape[0] != n_c:
        problems[m] = (f"the corpus matrix has {A.shape[0]} rows, its metadata "
                       f"says {n_c}")
    elif S.shape[0] != n_s:
        problems[m] = (f"the reference matrix has {S.shape[0]} rows, its metadata "
                       f"says {n_s}")
    elif n_s and A.shape[1] != S.shape[1]:
        problems[m] = (f"answers are {A.shape[1]} numbers wide and reference "
                       f"passages {S.shape[1]}: they are not in the same space, so "
                       f"a cosine between them is not a similarity")
    elif n_c != len(df):
        problems[m] = (f"{n_c} corpus rows against {len(df)} responses in "
                       f"openends_long.csv")
    elif list(meta.get("corpus_keys", [])) != corpus_keys:
        problems[m] = "different responses, or a different row order, from openends_long.csv"
    elif meta.get("run_mode") != RUN_MODE:
        problems[m] = f"written in RUN_MODE={meta.get('run_mode')}, this run is {RUN_MODE}"
    else:
        loaded[m] = (A, S, meta)
        print(f"  {m}: {n_c} corpus rows, {n_s} reference rows, width "
              f"{meta.get('dim')}, checkpoint {meta.get('checkpoint')}")
for m, why in problems.items():
    print(f"  [skip] {m}: {why}.")
    print(f"         Delete embeddings_{m}.npy and re-run notebook 08 with "
          f"ENCODER_KEY = '{m}'.")


  [ok]   embeddings_bge.npy  (written unknown time)
  [ok]   embeddings_bge_seed.npy  (written unknown time)
  [ok]   embeddings_bge_meta.json  (written unknown time)
  bge: 2844 corpus rows, 133 reference rows, width 1024, checkpoint BAAI/bge-large-en-v1.5
  [ok]   embeddings_mpnet.npy  (written unknown time)
  [ok]   embeddings_mpnet_seed.npy  (written unknown time)
  [ok]   embeddings_mpnet_meta.json  (written unknown time)
  mpnet: 2844 corpus rows, 133 reference rows, width 768, checkpoint sentence-transformers/all-mpnet-base-v2
  [ok]   embeddings_voyage.npy  (written unknown time)
  [ok]   embeddings_voyage_seed.npy  (written unknown time)
  [ok]   embeddings_voyage_meta.json  (written unknown time)
  voyage: 2844 corpus rows, 133 reference rows, width 1024, checkpoint voyage-3-large


In [16]:
# === THE REQUIRED LEG, AND THE REFERENCE ROWS ===
# BGE is the required encoder. With no BGE there is no primary encoder, hence no
# coverage flag and no embedding report at all, so a missing BGE stops the notebook
# here with a sentence saying what to run rather than producing a half-table that
# looks like a result.
if PRIMARY_MODEL not in loaded:
    for _f in (f"embeddings_{PRIMARY_MODEL}.npy",
               f"embeddings_{PRIMARY_MODEL}_seed.npy",
               f"embeddings_{PRIMARY_MODEL}_meta.json"):
        need(_f, produced_by="08")
    raise MissingInput(
        f"\n\n  The {PRIMARY_MODEL} files are present but unusable: "
        f"{problems.get(PRIMARY_MODEL)}.\n"
        f"  Delete embeddings_{PRIMARY_MODEL}.npy and run notebook 08 with "
        f"ENCODER_KEY = '{PRIMARY_MODEL}'.\n")

# The facet label of every reference row travels in the metadata, in the order
# notebook 08 embedded them. Taking the labels from THERE rather than re-deriving
# them from the pool file is what makes the slice safe: a rebuilt order would be one
# dictionary-ordering change away from labelling the vectors wrongly, and nothing
# would look broken.
_meta0 = loaded[PRIMARY_MODEL][2]
seed_frame = pd.DataFrame(_meta0.get("seed_rows", []),
                          columns=["facet", "response_id", "item_type", "source"])

def _keys(frame):
    """(facet, response_id, item_type) as text, for comparing two orderings."""
    return [tuple(map(str, r)) for r in
            frame[["facet", "response_id", "item_type"]].values]

# If notebook 04 rebuilt the pools after notebook 08 ran, these vectors describe
# passages that are no longer the reference sets. The shapes would still line up
# and every number below would still compute, which is why this is checked
# explicitly rather than left to a row count.
assert _keys(seed_frame) == _keys(pool_index), (
    "The reference passages embedded in embeddings_*.npy are not the ones in "
    "embedding_seed_pools.json. Notebook 04 has been re-run since notebook 08; "
    "re-run notebook 08 for every encoder.")

# THE CHECK ABOVE CANNOT SEE A REWRITTEN PASSAGE. Change the text of a reference
# passage and leave its ids alone and the comparison still passes, while every
# vector below describes a passage nobody would recognise. Notebook 04 stamps the
# pool file with the moment it built it, and notebook 08 copies that stamp into its
# metadata, so the two timestamps are what close the gap the ids leave open.
_built_now  = _prov.get("built_at")
_built_then = (_meta0.get("seed_pool_provenance") or {}).get("built_at")
if _built_now and _built_then:
    assert _built_now == _built_then, (
        f"embedding_seed_pools.json was built at {_built_now}, but "
        f"embeddings_{PRIMARY_MODEL}_meta.json says it embedded the pools built at "
        f"{_built_then}. The reference vectors describe passages that are no longer "
        f"the reference sets. Re-run notebook 08 for every encoder.")
    print(f"  reference vectors match the pool file built at {_built_now}")
else:
    print("  NOTE: no built_at stamp on one side of the comparison, so the "
          "reference passages are matched by id alone. A pool file rewritten in "
          "place, with the same ids, could not be detected here.")
print(f"  primary encoder: {PRIMARY_MODEL}; reference rows: {len(seed_frame)}")
print(f"  optional encoders: {[m for m in loaded if m != PRIMARY_MODEL] or 'none'}")


  reference vectors match the pool file built at 2026-08-13T22:09:53
  primary encoder: bge; reference rows: 133
  optional encoders: ['mpnet', 'voyage']


In [17]:
# === EACH ENCODER'S TWO MATRICES, CHECKED AGAINST EACH OTHER AND NORMALIZED ===
# The corpus and the reference passages arrive as separate files, so there is no
# block to cut here. What has to be established instead is that each encoder's pair
# belongs together, and that every encoder carries the SAME reference passages in
# the SAME order.
# WHY THAT SECOND POINT MATTERS: the whole purpose of the sensitivity table below
# is to compare encoders. Two encoders scored against different reference sets
# would differ for that reason alone, and the table would report an encoder effect
# that is really a pools effect.
emb = {}
for m, (A, S, meta) in loaded.items():
    _sf = pd.DataFrame(meta.get("seed_rows", []),
                       columns=["facet", "response_id", "item_type", "source"])
    # The metadata LISTS the reference passages; the reference matrix holds their
    # vectors. Where the two disagree on how many there are, every facet mask below
    # would select the wrong columns of the cosine matrix and the numbers would
    # still look ordinary. Checked once per encoder, because notebook 08 writes
    # each encoder's matrix and metadata as its own pair of files.
    if len(_sf) != S.shape[0]:
        print(f"  [skip] {m}: its metadata lists {len(_sf)} reference passages but "
              f"its reference matrix has {S.shape[0]} rows. Re-run notebook 08 "
              f"with ENCODER_KEY = '{m}'.")
        continue
    if _keys(_sf) != _keys(seed_frame):
        print(f"  [skip] {m}: its reference passages differ from {PRIMARY_MODEL}'s, "
              f"so the two cannot be compared. Re-run notebook 08 for {m}.")
        continue
    _b = (meta.get("seed_pool_provenance") or {}).get("built_at")
    if _b and _built_then and _b != _built_then:
        print(f"  [skip] {m}: it embedded the pools built at {_b}, while "
              f"{PRIMARY_MODEL} embedded the pools built at {_built_then}, so the "
              f"two are not comparable. Re-run notebook 08 for {m}.")
        continue
    # normalize_rows is applied even though notebook 08 wrote unit-length rows: it
    # is a no-op on rows that are already unit length, and it costs a fraction of a
    # second to stop trusting a flag written in another notebook.
    emb[m] = (normalize_rows(A), normalize_rows(S))
    print(f"  {m}: answers {emb[m][0].shape}, reference passages {emb[m][1].shape}")

# The primary encoder is dropped by the first check above whenever its own two
# files disagree about how many reference passages there are, and that is the state
# this line stops on: without the primary there is no coverage flag and no encoder
# of record, so there is nothing to report.
assert PRIMARY_MODEL in emb, (
    f"{PRIMARY_MODEL} was dropped by the checks above, so there is no primary "
    f"encoder and nothing can be reported. Re-run notebook 08 with "
    f"ENCODER_KEY = '{PRIMARY_MODEL}'.")
if len(seed_frame) == 0:
    # A real state, not an error: notebook 04 found no admissible passages at all.
    # Every similarity comes back NaN and the table reports the leg as not run.
    print("  NOTE: no reference passages at all. Every facet similarity will be "
          "NaN, and the whole embedding leg is NOT RUN rather than failed.")


  bge: answers (2844, 1024), reference passages (133, 1024)
  mpnet: answers (2844, 768), reference passages (133, 768)
  voyage: answers (2844, 1024), reference passages (133, 1024)


In [18]:
# === SCORE THE CORPUS, ONE ENCODER AT A TIME ===
# The encoders differ ONLY in the vectors they produce: identical scoring code, one
# function, called the same way for each. Any per-encoder difference in the
# sensitivity table below is then attributable to the encoder itself and not to
# three slightly divergent copies of the scoring code.
sims = {}
for m, (R, S) in emb.items():
    sim = facet_similarity(R, df["response_id"].values, df["item_type"].values,
                           seed_frame, S)
    # Namespace the columns by encoder so all the frames sit side by side below
    # without collision: desert_sim -> bge_desert_sim.
    sim = sim.rename(columns={f"{f}_sim": f"{m}_{f}_sim" for f in FACETS})
    sims[m] = sim
    print(f"\n-- {m}: per-facet similarity computed")
    # READ THIS OUTPUT AS: nine rows, one per facet, giving the mean and the
    # standard deviation of the reference-set cosine. The MEANS are encoder-specific
    # and are NOT comparable across encoders -- that shift IS the finding reported
    # in part (a) of the sensitivity table, not a defect. What matters here is the
    # SD: an SD at about 0 means every response sits the same distance from that
    # facet's set, so the column can order nothing and the facet carries no
    # embedding signal under this encoder. A row of "--" is a facet with no
    # reference passages in any item_type: not measured, rather than measured at
    # zero. na_rep is what prints it that way; without it pandas prints "NaN",
    # which reads as a failed computation instead of an absent one.
    print(sim.filter(like="_sim").describe().T[["mean", "std"]].round(3)
             .to_string(na_rep="--"))



-- bge: per-facet similarity computed
                          mean    std
bge_desert_sim           0.630  0.081
bge_proportionality_sim  0.598  0.066
bge_censure_sim          0.579  0.064
bge_moral_balance_sim    0.644  0.092
bge_revenge_sim          0.608  0.072
bge_suffering_sim        0.567  0.058
bge_deterrence_sim       0.626  0.105
bge_incapacitation_sim   0.626  0.098
bge_rehabilitation_sim   0.604  0.096

-- mpnet: per-facet similarity computed
                            mean    std
mpnet_desert_sim           0.431  0.140
mpnet_proportionality_sim  0.431  0.151
mpnet_censure_sim          0.374  0.113
mpnet_moral_balance_sim    0.441  0.172
mpnet_revenge_sim          0.399  0.136
mpnet_suffering_sim        0.362  0.103
mpnet_deterrence_sim       0.416  0.193
mpnet_incapacitation_sim   0.411  0.166
mpnet_rehabilitation_sim   0.415  0.160

-- voyage: per-facet similarity computed
                             mean    std
voyage_desert_sim           0.654  0.103
voyage_proportio

In [19]:
# === MODEL SENSITIVITY, PART 1: what is being compared with what ===
# WHAT: sets up the per-facet sensitivity table the preregistration asks for.
# PREREG 6.7: "As in the Dark Side study, the absolute level of the similarity
#       scores shifts across embedding models while the ordering of responses stays
#       stable, and we report that for the facet scores as well." The table
#       therefore has exactly two halves:
#         (a) the ABSOLUTE mean similarity per encoder -- expected to move, and
#             reported openly as moving;
#         (b) the cross-encoder RANK-ORDER (Spearman) correlation of the response
#             ordering -- expected to hold.
#       In the Dark Side pipeline the absolute metric swung from 67.5 per cent on
#       mpnet to 21 to 28 per cent on BGE and Voyage while the rank order held at
#       about .71. A number that moves forty points with the choice of encoder
#       cannot anchor anything; an ORDERING that survives the swap can support a
#       convergence check.
# THE RULE THIS EXISTS TO ENFORCE: only rank-stable facets enter convergence. A
#       facet whose ordering does not survive an encoder swap is measuring the
#       encoder, not the construct, and is reported as such rather than used.
# STILL NOT A GATE: enters_convergence admits a facet to the downstream convergence
#       REPORT. It passes and fails nothing.
# (response_id, item_type) is the join key everywhere below -- the same unit the
# similarity was computed at, so nothing is aggregated at this step.
model_sims = {m: sims[m].set_index(["response_id", "item_type"]) for m in sims}
MODELS = list(model_sims.keys())
# All unordered encoder pairs: three encoders give three pairs, two give one, one
# gives none.
pairs = [(MODELS[i], MODELS[j])
         for i in range(len(MODELS)) for j in range(i + 1, len(MODELS))]
print(f"Encoders in the sensitivity table: {MODELS}")

# THE CACHE MIX-UP, TESTED RATHER THAN LEFT AS A NOTE IN THE MARGIN. If notebook 08
# ever wrote one encoder's vectors under another encoder's name, the two similarity
# columns would be identical, every rank correlation would read 1.000, and the most
# reassuring table this notebook can print would be an artefact. Comparing the
# response vectors themselves is what catches it; comparing the recorded checkpoint
# strings would not, because a mix-up copies the numbers and leaves the label.
identical_encoders = [(a, b) for a, b in pairs
                      if emb[a][0].shape == emb[b][0].shape
                      and np.array_equal(emb[a][0], emb[b][0])]
if identical_encoders:
    print("  WARNING: these encoder pairs hold IDENTICAL response vectors, so the")
    print("  rank correlations below compare a column with itself and mean nothing:")
    for a, b in identical_encoders:
        print(f"    {a} and {b}")
    print("  Delete both .npy files and re-run notebook 08 for each encoder.")
elif pairs:
    print("  checked: no two encoders hold the same response vectors.")

# HAZARD: with one encoder there are no PAIRS, so there is no rank-stability
# evidence to be had, rank_stable is False for every facet, and nothing enters
# convergence. That is the correct outcome -- one encoder cannot demonstrate that
# an ordering survives changing the encoder -- but it makes an all-False table a
# statement about the RUN, not about the facets. Run at least two encoders before
# reading anything into it.
if not pairs:
    print("  NOTE: one encoder only, so rank stability is NOT RUN for every facet.")


Encoders in the sensitivity table: ['bge', 'mpnet', 'voyage']
  checked: no two encoders hold the same response vectors.


In [20]:
# === MODEL SENSITIVITY, PART 2: the coverage floor ===
def _nonzero_variance(series):
    """Does this facet's similarity column carry variance worth ordering?

    A floor on the DISTRIBUTION, and not a coverage statistic. It asks only
    whether the column varies enough to order responses at all. Whether the facet
    is present in the corpus often enough for a convergence claim is a different
    question, put to the word list in notebook 02 and to the ensemble's realized
    base rate in notebook 10. True here is therefore not evidence that a sparse
    facet is adequately covered: it says the column is not degenerate, and it says
    nothing else.
    """
    # Two conditions, both deliberately weak -- this is a floor, not a test.
    #   len >= 10  : a variance computed on fewer than ten scored responses is noise.
    #   std > 1e-6 : a flat column means every response sits the same distance from
    #                the facet's reference set, so the column can order nothing.
    # This is where the sparse facets are expected to fail -- proportionality runs
    # about 1.0 per cent in the corpus (0.3 per cent on the backward item), and
    # censure, moral_balance and revenge are comparably thin. The word list carries
    # a parallel rule for the same reason: below 5 per cent coverage it is
    # disqualified as a convergence leg, because per-facet agreement computed on
    # three to thirteen positives is noise whose silence reads falsely as "no
    # independent contradiction" when it is really "no independent measurement".
    s = pd.to_numeric(series, errors="coerce").dropna()
    return len(s) >= 10 and s.std() > 1e-6

# The frozen split matters under the registered default too, even though no
# embedding-versus-human number is computed there: it is what kept the reference
# passages off the TEST half in the first place, audited near the top of this
# notebook.
if HUMAN_GOLD:
    # TEST-half membership, counted only so the line below can state the TEST n.
    # Nothing in this notebook conditions on it: the sensitivity table is a
    # property of the ENCODERS across the whole corpus, not a validation number,
    # and saying so in the output is what keeps it from being read as one.
    test_units = set(map(tuple, split.loc[split.gold_split == "TEST",
                                          ["response_id", "item_type"]].values))
    print(f"  Embedding-versus-human agreement is TEST-only and runs in the "
          f"optional human-coding notebook (TEST n={len(test_units)} "
          f"response x item rows).")
else:
    print("  Embedding-versus-human agreement: NOT RUN. HUMAN_GOLD is False, which "
          "is the registered pipeline (prereg 6.7 prescribes no human-coding step).")


  Embedding-versus-human agreement: NOT RUN. HUMAN_GOLD is False, which is the registered pipeline (prereg 6.7 prescribes no human-coding step).


In [21]:
# === MODEL SENSITIVITY, PART 3: (a) absolute means and (b) rank correlations ===
rows = []
# Why a pair could not be correlated, per facet, so the table can give the REASON
# rather than a bare blank. Recorded here, read in the next cell.
pair_notes = {f: [] for f in FACETS}
for f in FACETS:
    col = f"{f}_sim"
    # sparse_facet and the passage count ride along in the export so a reader can
    # see at a glance which rows are expected to be thin, and which have no
    # reference set at all, before reading any of their numbers.
    rec = {"facet": f, "sparse_facet": f in SPARSE_FACETS,
           "n_reference_passages": int((seed_frame["facet"].values == f).sum())}
    # (a) ABSOLUTE mean similarity per encoder -- expected to shift. Reported
    # precisely so that the shift is visible. These columns must NOT be compared
    # across encoders as though they were the same quantity, and no threshold
    # anywhere in this pipeline is ever set on an absolute cosine.
    for m in MODELS:
        rec[f"{m}_abs_mean"] = pd.to_numeric(model_sims[m][f"{m}_{col}"],
                                             errors="coerce").mean()
    # (b) cross-encoder RANK-ORDER correlation of the response ordering -- expected
    # to hold.
    for m1, m2 in pairs:
        # Inner join on (response_id, item_type): compare two encoders only on the
        # responses BOTH scored, so a difference in the NaN pattern cannot present
        # itself as disagreement about ordering.
        j = model_sims[m1][[f"{m1}_{col}"]].join(
            model_sims[m2][[f"{m2}_{col}"]], how="inner").dropna()
        if len(j) < 10:
            # Too few jointly scored responses to say anything: NaN, not 0.
            rho = np.nan
            pair_notes[f].append("too few jointly scored responses")
        elif (j[f"{m1}_{col}"].nunique() < 2) or (j[f"{m2}_{col}"].nunique() < 2):
            # A flat column has no ordering to correlate, so scipy returns NaN and
            # warns. Naming the case here is what stops the table reporting "too
            # few responses" for a facet that had thousands and simply did not vary.
            rho = np.nan
            pair_notes[f].append("at least one encoder's column is flat, "
                                 "so it orders nothing")
        else:
            # Spearman (rank), not Pearson (linear). The claim under test is about
            # ORDERING. A monotone but non-linear re-scaling between encoders is
            # precisely what (a) already reports, and it is not disagreement.
            rho, _ = spearmanr(j[f"{m1}_{col}"], j[f"{m2}_{col}"])
        rec[f"rank_stab_{m1}_{m2}"] = rho
    rows.append(rec)
print(f"  computed (a) absolute means for {len(MODELS)} encoder(s) and "
      f"(b) rank correlations for {len(pairs)} pair(s), across {len(FACETS)} facets.")


  computed (a) absolute means for 3 encoder(s) and (b) rank correlations for 3 pair(s), across 9 facets.


In [22]:
# === MODEL SENSITIVITY, PART 4: the two flags, and saying "not run" when it is ===
for rec in rows:
    col = f"{rec['facet']}_sim"
    # Coverage floor, checked on the REQUIRED encoder only. A facet that is
    # degenerate under the primary has no primary embedding measurement to report,
    # whatever an optional encoder did.
    rec[f"{PRIMARY_MODEL}_nonzero_variance"] = _nonzero_variance(
        model_sims[PRIMARY_MODEL][f"{PRIMARY_MODEL}_{col}"])
    rho_vals = [rec[f"rank_stab_{a}_{b}"] for a, b in pairs
                if pd.notna(rec[f"rank_stab_{a}_{b}"])]
    # Where the rank half could not be computed, the table SAYS so, and says WHY.
    # A blank cell or a bare NaN here would be read as a facet that failed, when
    # what happened is that the check never ran.
    if rec["n_reference_passages"] == 0:
        rec["rank_stability"] = "not run -- no reference passages for this facet"
    elif not pairs:
        rec["rank_stability"] = "not run -- only one encoder available"
    elif not rho_vals:
        _why = sorted(set(pair_notes[rec["facet"]])) or ["no pair could be computed"]
        rec["rank_stability"] = "not run -- " + "; ".join(_why)
    else:
        rec["rank_stability"] = "computed"
    # rank-stable = every available pairwise Spearman >= .50. MIN across pairs, not
    # mean: one disagreeing pair is enough to show the ordering is not
    # encoder-invariant, and an average would let a strong pair hide a weak one.
    # The .50 bar is an implementer's choice, not a value the preregistration or the
    # design fixes -- Dark Side rank stability ran about .71. It is exported below
    # with the raw rho values beside it precisely so a reader can re-threshold;
    # ratify or move it with the model roster (design section 11).
    rec["rank_stable"] = bool(rho_vals) and min(rho_vals) >= 0.50
    # BOTH conditions: the ordering must survive the encoder swap AND the column
    # must carry variance worth ordering. Read this flag for what it is -- admission
    # to the downstream convergence REPORT, never a pass or a fail, and never read
    # by any gate.
    rec["enters_convergence"] = bool(rec["rank_stable"]
                                     and rec[f"{PRIMARY_MODEL}_nonzero_variance"])

sensitivity = pd.DataFrame(rows)
print(f"  sensitivity table built: {len(sensitivity)} facets, "
      f"{sensitivity.enters_convergence.sum()} entering the convergence report.")


  sensitivity table built: 9 facets, 9 entering the convergence report.


In [23]:
# === THE TABLE, AND HOW TO READ IT ===
print("\n=== Per-facet model-sensitivity table ===")
# na_rep="--" so an uncomputable cell prints as a dash. Without it pandas prints
# "NaN", which reads as a number that failed rather than as a quantity that was
# never defined -- and the rank_stability column beside it gives the reason.
print(sensitivity.round(3).to_string(index=False, na_rep="--"))
# HOW TO READ IT. The *_abs_mean columns SHOULD differ across encoders -- that is
# finding (a), and it is why no absolute similarity is ever a headline number. The
# rank_stab_* columns should be high and close to one another -- that is finding
# (b). A facet with high rank_stab and nonzero variance under the primary encoder
# is one whose ordering means something whichever encoder produced it, and only
# those reach the convergence report.
# DIAGNOSTICS FOR AN EMPTY-LOOKING RUN: rank_stability all "not run" means only one
# encoder ran; nonzero_variance all False means the reference pools did not load and
# every similarity is NaN. Two encoders sharing one set of vectors -- a cache
# mix-up in notebook 08, which would make every rank_stab read 1.000 -- is checked
# for above rather than left to a reader noticing two identical float columns.
if identical_encoders:
    print("\nDO NOT READ THE rank_stab COLUMNS: the encoder pairs "
          f"{identical_encoders} hold identical vectors (see the check above).")
print("\nSparse facets are counted only where the primary encoder shows nonzero variance.")
print("Nothing in this table gates a facet. The registered gate is inter-model "
      "ICC >= .60 plus facet r >= .30 with the matching survey scale, and neither "
      "clause is computed from an embedding.")



=== Per-facet model-sensitivity table ===
          facet  sparse_facet  n_reference_passages  bge_abs_mean  mpnet_abs_mean  voyage_abs_mean  rank_stab_bge_mpnet  rank_stab_bge_voyage  rank_stab_mpnet_voyage  bge_nonzero_variance rank_stability  rank_stable  enters_convergence
         desert         False                    15         0.630           0.431            0.654                0.833                 0.800                   0.856                  True       computed         True                True
proportionality          True                    15         0.598           0.431            0.620                0.732                 0.719                   0.805                  True       computed         True                True
        censure          True                    15         0.579           0.374            0.611                0.731                 0.622                   0.724                  True       computed         True                True
  moral_balan

In [24]:
# === RESHAPE TO LONG FORM FOR THE DOWNSTREAM JOIN ===
# WHY LONG: the convergence notebook joins similarity to the ensemble scores and the
# word-list scores at the (response_id, item_type, facet) grain. Long form makes
# that a join rather than nine column-name manipulations, and it keeps the file's
# shape stable when an encoder is added or dropped -- that changes one column, not
# nine.
long_parts = []
for m in MODELS:
    wide = model_sims[m].reset_index()
    melted = wide.melt(id_vars=["response_id", "item_type"],
                       value_vars=[f"{m}_{f}_sim" for f in FACETS],
                       var_name="facet", value_name=f"{m}_sim")
    # Strip the encoder prefix and the _sim suffix back off, so bge_moral_balance_sim
    # becomes the bare facet moral_balance and every long frame shares one index.
    # regex=False so that a facet name is never read as a pattern.
    melted["facet"] = (melted["facet"].str.replace(f"{m}_", "", regex=False)
                                      .str.replace("_sim", "", regex=False))
    long_parts.append(melted.set_index(["response_id", "item_type", "facet"]))

# Concatenate on axis=1 over the shared (response_id, item_type, facet) index: one
# row per unit, one column per encoder. Aligning on the INDEX rather than on
# position is what guarantees a Voyage score can never land on a BGE row. It works
# only because (response_id, item_type) is unique in openends_long.csv, which is
# not assumed here: it is asserted in the Load cell above, on the file as read.
sim_long = pd.concat(long_parts, axis=1).reset_index()
print(f"  long form: {len(sim_long)} rows, columns {list(sim_long.columns)}")


  long form: 25596 rows, columns ['response_id', 'item_type', 'facet', 'bge_sim', 'mpnet_sim', 'voyage_sim']


In [25]:
# === TAG EACH ROW WITH ITS FROZEN-SPLIT MEMBERSHIP ===
# how="left" keeps every response: only the rows in the frozen split carry a SEED
# or TEST label, and the rest are labelled NOT_IN_SPLIT -- spelled out rather than
# left blank, because a blank cell in a results file gets read as a failure when it
# means "this answer was never drawn into the split sample at all".
# CONTRACT: the split is frozen PER ITEM, so the merge takes BOTH keys. On
# response_id alone each of a participant's items would pick up every one of that
# participant's split rows: the export multiplies, and an item frozen into SEED can
# come back labelled TEST -- the exact thing the freeze exists to prevent. That the
# right-hand frame holds one row per (response_id, item_type) is asserted where the
# file is read, in the Load cell above.
n_sim_rows = len(sim_long)
sim_long = sim_long.merge(split[["response_id", "item_type", "gold_split"]],
                          on=["response_id", "item_type"], how="left")
# A LEFT join on a key proven unique cannot change the row count. This line is what
# catches the two ways that stops being true: the frame being merged acquiring
# duplicate keys after the check above, and this call being edited to an inner or
# outer join, either of which would silently reshape the export.
assert len(sim_long) == n_sim_rows, (
    f"The split merge changed the row count ({n_sim_rows} -> {len(sim_long)}). "
    "Either gold_split_assignment.csv gained duplicate response_id x item_type "
    "rows after it was read, or this merge is no longer a left join.")

# THE MERGE MUST ALSO HAVE MATCHED, and a row count is blind to one that matched
# NOTHING: every row would come back unlabelled, fillna would write NOT_IN_SPLIT
# across the whole file, and that value reads as a legitimate state rather than as
# a failure. Notebooks 10 and 11 both drop gold_split == "SEED" before computing
# anything, so a silently unmatched merge puts every seeded answer back into the
# numbers the frozen split exists to keep it out of (anti-circularity rule 4).
# WHAT IS EXPECTED TO MATCH: in a full run, every row of the split, because
# notebook 03 drew the sample FROM openends_long.csv. In smoke mode the corpus is
# 20 participants while the split still describes the whole sample, so only the
# overlap can match. The keys are compared as TEXT, which is what makes this an
# independent check rather than a restatement of the merge: a join that matched
# nothing because one file types response_id as a number and the other as text
# still fails here.
_corpus_units = set(zip(df.response_id.astype(str), df.item_type.astype(str)))
_split_units = set(zip(split.response_id.astype(str), split.item_type.astype(str)))
_expected = len(_split_units if RUN_MODE == "full"
                else _split_units & _corpus_units)
_labelled = sim_long.loc[sim_long.gold_split.notna(),
                         ["response_id", "item_type"]].astype(str).drop_duplicates()
assert len(_labelled) == _expected, (
    f"The split merge labelled {len(_labelled)} answers where {_expected} were "
    f"expected. Either gold_split_assignment.csv describes a different corpus, in "
    f"which case re-run notebook 03 against the current openends_long.csv, or the "
    f"two files disagree about the type or the spelling of response_id and "
    f"item_type. Left alone, every seeded answer re-enters the analyses the freeze "
    f"exists to keep it out of.")
sim_long["gold_split"] = sim_long["gold_split"].fillna("NOT_IN_SPLIT")
print(f"  frozen-split labels attached: "
      f"{sim_long.gold_split.value_counts().to_dict()}")


  frozen-split labels attached: {'NOT_IN_SPLIT': 22824, 'SEED': 1386, 'TEST': 1386}


## Save and check

In [26]:
# === SAVE: two files, both through done() ===
# THE EXPORT CONTRACT, CHECKED BEFORE THE FILE IS WRITTEN, not asserted in a
# comment. One row per (response_id, item_type, facet): that triple is the key
# notebook 12 declares for this file and the grain notebooks 10 and 11 join on. A
# duplicated row would raise no error anywhere downstream. It would raise a sample
# size while the n printed beside it stayed the same, and shrink every p-value
# computed from it.
_dup_out = sim_long[sim_long.duplicated(["response_id", "item_type", "facet"],
                                        keep=False)]
assert _dup_out.empty, (
    f"facet_similarity.csv would ship {len(_dup_out)} duplicated "
    f"(response_id, item_type, facet) rows, for example "
    f"{_dup_out[['response_id', 'item_type', 'facet']].head(3).to_dict('records')}.")
# And exactly one row per answer per facet. This is an identity, not an upper
# bound: notebooks 00 and 08 keep a blank answer as a row rather than dropping it,
# so a participant who left an open-end empty still contributes three rows and nine
# facet scores per row. Fewer than the product means rows were lost in the reshape;
# more means they were duplicated.
assert len(sim_long) == len(df) * len(FACETS), (
    f"facet_similarity.csv has {len(sim_long)} rows; {len(df)} answers x "
    f"{len(FACETS)} facets is {len(df) * len(FACETS)}.")

sim_long.to_csv("facet_similarity.csv", index=False)
done("facet_similarity.csv", rows=len(sim_long),
     note=f"reference-set mean cosine, per item_type, leave-one-out; "
          f"encoders={MODELS}")

# The raw rho values are exported beside the flags so that the .50 rank-stability
# bar is auditable rather than buried in code.
sensitivity.to_csv("embedding_model_sensitivity.csv", index=False)
done("embedding_model_sensitivity.csv", rows=len(sensitivity),
     note="(a) absolute mean similarity per encoder, (b) cross-encoder rank correlations")

print("\nNOTEBOOK 09 -- what was produced")
print(f"  encoders scored: {MODELS}")
print(f"  facet_similarity.csv: {len(sim_long)} rows = {len(df)} answers x "
      f"{len(FACETS)} facets, one similarity column per encoder, plus the "
      f"frozen-split label.")
print(f"  embedding_model_sensitivity.csv: {len(sensitivity)} facets; "
      f"{int(sensitivity.enters_convergence.sum())} enter the convergence report.")
print("  Reference passages: SEED half only, audited above.")
print("  This leg gates nothing. The registered gate is inter-model ICC >= .60 and")
print("  facet r >= .30 with the matching survey scale; no embedding enters either.")
print("  Next: the convergence notebook, which joins this file to the ensemble and")
print("  word-list scores on (response_id, item_type, facet).")


  [wrote] facet_similarity.csv  (25596 rows, 2,589,062 bytes)
  [wrote] embedding_model_sensitivity.csv  (9 rows, 1,632 bytes)

NOTEBOOK 09 -- what was produced
  encoders scored: ['bge', 'mpnet', 'voyage']
  facet_similarity.csv: 25596 rows = 2844 answers x 9 facets, one similarity column per encoder, plus the frozen-split label.
  embedding_model_sensitivity.csv: 9 facets; 9 enter the convergence report.
  Reference passages: SEED half only, audited above.
  This leg gates nothing. The registered gate is inter-model ICC >= .60 and
  facet r >= .30 with the matching survey scale; no embedding enters either.
  Next: the convergence notebook, which joins this file to the ensemble and
  word-list scores on (response_id, item_type, facet).
